In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [48]:
!pip install --upgrade gdown

Defaulting to user installation because normal site-packages is not writeable


In [1]:
!pip install spectral

In [2]:
!pip install --user opencv-python


In [4]:
!pip install matplotlib


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 40.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 36.3 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [7]:
!pip install torch torchvision torchaudio


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 61.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 68.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 63.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 68.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 79.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 59.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 53.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 

In [2]:
rm -rf /home/aaol232/.local/share/Trash

In [ ]:
import os
import pandas as pd

# Define individual paths
hsi_dir = '/mnt/gpfs2_4m/scratch/VVV/Project 1/Reconstruction-Wheat'
rgb_dir = '/mnt/gpfs2_4m/scratch/VVV/Project 1/Reconstruction-Wheat/RGB-Reconstruction-Wheat'
csv_file_path = '/mnt/gpfs2_4m/scratch/VVV/Project 1/Wavelength_Band_VIS1.csv'

# List files in each directory
print("HSI files:", os.listdir(hsi_dir))
print("RGB files:", os.listdir(rgb_dir))

# Load CSV
df = pd.read_csv(csv_file_path)
print(df.head())

In [ ]:
import os
import numpy as np
import spectral.io.envi as envi
import cv2
import gc
from skimage.filters import threshold_otsu, gaussian
from skimage.measure import label

# Helper functions
def get_sample_files(dataset_dir):
    return [os.path.join(dataset_dir, f.replace(".hdr", ""))
            for f in os.listdir(dataset_dir)
            if f.endswith(".hdr") and "Dark Reference" not in f]

def get_rgb_filename(sample_file):
    base_name = os.path.basename(sample_file)
    base_name = '-'.join(base_name.split('-')[:-1])
    return rgb_files.get(base_name, None)

def apply_post_processing(rgb_image):
    rgb_image = (rgb_image - np.min(rgb_image)) / (np.max(rgb_image) - np.min(rgb_image)) * 255
    rgb_image = rgb_image.astype(np.uint8)
    gamma = 2.2
    inv_gamma = 1.0 / gamma
    gamma_table = np.array([(i / 255.0) ** inv_gamma * 255 for i in np.arange(256)]).astype(np.uint8)
    rgb_image = cv2.LUT(rgb_image, gamma_table)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    for i in range(3):
        rgb_image[:, :, i] = clahe.apply(rgb_image[:, :, i])
    return rgb_image

def extract_reflectance_cube_cropped_roi(sample_file, dark_file, wavelengths_of_interest,
                                         band_for_mask=1000, smooth_sigma=1, min_region_size=200):
    sample_img = envi.open(sample_file + '.hdr', sample_file + '.raw').load().astype(np.float32)
    dark_img = envi.open(dark_file + '.hdr', dark_file + '.raw').load().astype(np.float32)
    rows, cols, bands = sample_img.shape

    ref_band = sample_img[:, :, band_for_mask]
    thresh_val = threshold_otsu(ref_band)
    mask = ref_band > thresh_val
    smoothed = gaussian(mask, sigma=smooth_sigma)
    thresh_mask = smoothed > 0.04
    labeled_img = label(thresh_mask, connectivity=2)

    regions = [(lbl, np.sum(labeled_img == lbl)) for lbl in np.unique(labeled_img)
               if lbl != 0 and np.sum(labeled_img == lbl) > min_region_size]
    if len(regions) < 2:
        raise ValueError("Not enough distinct regions found.")

    regions_sorted = sorted(regions, key=lambda x: x[1])
    white_label = regions_sorted[0][0]
    roi_label = regions_sorted[1][0]

    mask_white = (labeled_img == white_label)
    mask_corn = (labeled_img == roi_label)

    white_reflectance = np.zeros((cols, bands), dtype=np.float32)
    dark_reflectance = np.zeros((cols, bands), dtype=np.float32)

    for c in range(cols):
        white_rows = np.where(mask_white[:, c])[0]
        if white_rows.size > 0:
            white_reflectance[c, :] = np.mean(sample_img[white_rows, c, :], axis=0)
        else:
            white_reflectance[c, :] = np.nan
        dark_reflectance[c, :] = np.mean(dark_img[:, c, :], axis=0)

    corn_coords = np.argwhere(mask_corn)
    r_min, r_max = corn_coords[:, 0].min(), corn_coords[:, 0].max()
    c_min, c_max = corn_coords[:, 1].min(), corn_coords[:, 1].max()

    roi_height = r_max - r_min + 1
    roi_width = c_max - c_min + 1

    reflectance_cube_roi = np.full((roi_height, roi_width, len(wavelengths_of_interest)), np.nan, dtype=np.float32)
    denom_all = white_reflectance - dark_reflectance

    for (r, c) in corn_coords:
        denom = denom_all[c, :]
        pixel_vals = sample_img[r, c, :]
        dark_vals = dark_reflectance[c, :]
        valid = (denom != 0) & (~np.isnan(denom))
        if np.any(valid):
            pixel_refl = np.full(bands, np.nan, dtype=np.float32)
            pixel_refl[valid] = (pixel_vals[valid] - dark_vals[valid]) / denom[valid]
            rb, cb = r - r_min, c - c_min
            for i, w_idx in enumerate(wavelengths_of_interest):
                reflectance_cube_roi[rb, cb, i] = pixel_refl[w_idx]

    return reflectance_cube_roi

def process_dataset(dataset_dir, wavelengths_of_interest, new_rgb_save_dir, new_hsi_save_dir, batch_size=5):
    os.makedirs(new_rgb_save_dir, exist_ok=True)
    os.makedirs(new_hsi_save_dir, exist_ok=True)

    dark_ref_file = os.path.join(dataset_dir, "Dark Reference-WF-CF-4_1_25")
    sample_files = get_sample_files(dataset_dir)

    for i in range(0, len(sample_files), batch_size):
        batch = sample_files[i:i + batch_size]
        for sample_file in batch:
            try:
                hsi_image = extract_reflectance_cube_cropped_roi(sample_file, dark_ref_file, wavelengths_of_interest)
                rgb_file_name = get_rgb_filename(sample_file)
                if rgb_file_name is None:
                    print(f"No matching RGB file for {sample_file}")
                    continue

                rgb_image_path = os.path.join(rgb_images_dir, rgb_file_name)
                rgb_image = cv2.imread(rgb_image_path)

                if rgb_image is None:
                    print(f"Failed to read RGB image at path: {rgb_image_path}")
                    continue

                rgb_image = cv2.cvtColor(rgb_image, cv2.COLOR_BGR2RGB)
                processed_rgb = apply_post_processing(rgb_image)

                filename = os.path.splitext(rgb_file_name)[0]
                np.save(os.path.join(new_hsi_save_dir, f"{filename}.npy"), hsi_image)
                cv2.imwrite(os.path.join(new_rgb_save_dir, f"{filename}.jpg"), cv2.cvtColor(processed_rgb, cv2.COLOR_RGB2BGR))

                del hsi_image, rgb_image, processed_rgb
                gc.collect()

            except Exception as e:
                print(f"Error processing {sample_file}: {e}")

# === Updated paths ===
data_dir = '/mnt/gpfs2_4m/scratch/VVVV/Project 1/Reconstruction-Wheat'
rgb_images_dir = '/mnt/gpfs2_4m/scratch/VVVVV/Project 1/Reconstruction-Wheat/RGB-Reconstruction-Wheat'
new_rgb_save_dir = os.path.join(data_dir, 'processed_rgb_images')
new_hsi_save_dir = os.path.join(data_dir, 'processed_hsi_images')

# Map RGB filenames
rgb_files = {os.path.splitext(f)[0]: f for f in os.listdir(rgb_images_dir)}

# Wavelength indices of interest
wavelengths_of_interest = [102, 122, 124, 127, 150, 180, 185, 191, 630, 950]

# Run the processing
process_dataset(data_dir, wavelengths_of_interest, new_rgb_save_dir, new_hsi_save_dir)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

def extract_rgb_roi_kmeans(image_path, K=3, cluster_selection='largest_area'):
    bgr_img = cv2.imread(image_path)
    if bgr_img is None:
        raise IOError(f"Unable to read image from: {image_path}")

    bgr_img = cv2.GaussianBlur(bgr_img, (5, 5), 0)
    lab_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)

    l_channel, a_channel, b_channel = cv2.split(lab_img)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_channel = clahe.apply(l_channel)
    lab_img = cv2.merge([l_channel, a_channel, b_channel])

    rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)

    h, w = bgr_img.shape[:2]
    pixel_values = lab_img.reshape((-1, 3)).astype(np.float32)

    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    _, labels, centers = cv2.kmeans(pixel_values, K, None, criteria, 10, cv2.KMEANS_PP_CENTERS)

    labels_2d = labels.reshape((h, w))
    cluster_masks = [(labels_2d == i).astype(np.uint8) for i in range(K)]

    if cluster_selection == 'largest_area':
        cluster_areas = [mask.sum() for mask in cluster_masks]
        best_cluster_idx = np.argmax(cluster_areas)
    elif cluster_selection == 'greenest':
        best_cluster_idx = np.argmin(centers[:, 1])
    else:
        raise ValueError("Unknown cluster_selection method")

    chosen_mask = cluster_masks[best_cluster_idx]

    kernel = np.ones((5, 5), np.uint8)
    chosen_mask = cv2.morphologyEx(chosen_mask, cv2.MORPH_OPEN, kernel)
    chosen_mask = cv2.morphologyEx(chosen_mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(chosen_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        print("No contours found in chosen cluster.")
        return rgb_img, None, chosen_mask

    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w_box, h_box = cv2.boundingRect(largest_contour)

    roi_rgb = rgb_img[y:y+h_box, x:x+w_box, :]

    return rgb_img, (x, y, w_box, h_box), roi_rgb, chosen_mask


# -----------------------------
# Set your updated directories here
# -----------------------------
new_rgb_dir = '/mnt/gpfs2_4m/scratch/VVVVV/Project 1/Reconstruction-Wheat/processed_rgb_images' 

# Example usage:
image_name = "WF-CF-0.8-Rep7.jpg"  # Adjust filename as needed
image_path = os.path.join(new_rgb_dir, image_name)

rgb_image, bbox, roi, final_mask = extract_rgb_roi_kmeans(
    image_path,
    K=2,
    cluster_selection='greenest'  # or 'largest_area'
)

import matplotlib.patches as patches

if bbox is not None:
    x, y, w_box, h_box = bbox

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.title("Original Image with ROI Box")
    plt.imshow(rgb_image)
    ax = plt.gca()
    rect = patches.Rectangle((x, y), w_box, h_box, linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.title("Chosen Cluster Mask")
    plt.imshow(final_mask, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.title("Cropped ROI")
    plt.imshow(roi)
    plt.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("ROI extraction failed or no suitable cluster found.")

In [4]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

class HSI_RGB_Dataset(Dataset):
    def __init__(self, hsi_dir, rgb_dir, hsi_min, hsi_max, rgb_min, rgb_max, transform=None,
                 target_size=(512,512), rgb_interpolation=cv2.INTER_LINEAR, hsi_interpolation=cv2.INTER_LINEAR,
                 use_roi=True):
        self.hsi_dir = hsi_dir
        self.rgb_dir = rgb_dir
        self.transform = transform
        self.target_size = target_size
        self.rgb_interpolation = rgb_interpolation
        self.hsi_interpolation = hsi_interpolation
        self.use_roi = use_roi
        
        self.hsi_min, self.hsi_max = hsi_min, hsi_max
        self.rgb_min, self.rgb_max = rgb_min, rgb_max
        
        self.sample_names = sorted([os.path.splitext(f)[0] for f in os.listdir(hsi_dir) if f.endswith('.npy')])
    
    def __len__(self):
        return len(self.sample_names)
    
    def __getitem__(self, idx):
        sample_name = self.sample_names[idx]
        
        # Load HSI image from new_hsi_save_dir
        hsi_path = os.path.join(self.hsi_dir, f"{sample_name}.npy")
        hsi_image = np.load(hsi_path)
        hsi_image = np.nan_to_num(hsi_image, nan=0.1)
        hsi_image = (hsi_image - self.hsi_min) / (self.hsi_max - self.hsi_min)
        
        # Load RGB image from new_rgb_dir
        rgb_path = os.path.join(self.rgb_dir, f"{sample_name}.jpg")
        bgr_image = cv2.imread(rgb_path)
        if bgr_image is None:
            raise FileNotFoundError(f"RGB image not found: {rgb_path}")
        
        if self.use_roi:
            _, bbox, roi_rgb, _ = extract_rgb_roi_kmeans(rgb_path, K=2, cluster_selection='greenest')
            if roi_rgb is None:
                roi_rgb = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
        else:
            roi_rgb = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
        
        rgb_image = (roi_rgb - self.rgb_min) / (self.rgb_max - self.rgb_min)
        
        target_h, target_w = self.target_size
        hsi_resized = cv2.resize(hsi_image, (target_w, target_h), interpolation=self.hsi_interpolation)
        rgb_resized = cv2.resize(rgb_image, (target_w, target_h), interpolation=self.rgb_interpolation)
        
        if self.transform:
            from PIL import Image
            rgb_for_transform = (rgb_resized * 512).astype(np.uint8)
            rgb_pil = Image.fromarray(rgb_for_transform)
            rgb_tensor = self.transform(rgb_pil)
        else:
            rgb_tensor = torch.tensor(rgb_resized, dtype=torch.float32).permute(2, 0, 1)
        
        hsi_tensor = torch.tensor(hsi_resized, dtype=torch.float32).permute(2, 0, 1)
        
        return rgb_tensor, hsi_tensor,sample_name



In [ ]:
import os
import numpy as np

def compute_global_hsi_min_max(hsi_dir):
    global_min = float('inf')
    global_max = float('-inf')
    
    for file in os.listdir(hsi_dir):
        if file.endswith('.npy'):
            path = os.path.join(hsi_dir, file)
            hsi = np.load(path)
            hsi = np.nan_to_num(hsi, nan=0.1)
            cur_min = hsi.min()
            cur_max = hsi.max()
            if cur_min < global_min:
                global_min = cur_min
            if cur_max > global_max:
                global_max = cur_max
    return global_min, global_max

# Updated path to your processed HSI images directory
new_hsi_save_dir = '/mnt/gpfs2_4m/scratch/VVV/Project 1/Reconstruction-Wheat/processed_hsi_images'

# Compute global min and max
hsi_min, hsi_max = compute_global_hsi_min_max(new_hsi_save_dir)
print("Global HSI min:", hsi_min)
print("Global HSI max:", hsi_max)

In [ ]:
import os
import cv2

def compute_global_rgb_min_max(rgb_dir):
    global_min = float('inf')
    global_max = float('-inf')
    
    for file in os.listdir(rgb_dir):
        if file.endswith('.jpg'):
            path = os.path.join(rgb_dir, file)
            image = cv2.imread(path)
            if image is None:
                continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            cur_min = image.min()
            cur_max = image.max()
            if cur_min < global_min:
                global_min = cur_min
            if cur_max > global_max:
                global_max = cur_max
    return global_min, global_max

# Updated path to your processed RGB images directory
new_rgb_dir = '/mnt/gpfs2_4m/scratch/VVV/Project 1/Reconstruction-Wheat/processed_rgb_images'

rgb_min, rgb_max = compute_global_rgb_min_max(new_rgb_dir)
print("Global RGB min:", rgb_min)
print("Global RGB max:", rgb_max)

In [7]:
def collate_fn(batch):
    rgb_batch = torch.stack([item[0] for item in batch])
    hsi_batch = torch.stack([item[1] for item in batch])
    sample_names = [item[2] for item in batch]
    return rgb_batch, hsi_batch, sample_names

from torch.utils.data import DataLoader, random_split

# Updated directories
new_hsi_save_dir = '/mnt/gpfs2_4m/scratch/VVVV/Project 1/Reconstruction-Wheat/processed_hsi_images'
new_rgb_dir = '/mnt/gpfs2_4m/scratch/vvVVV/Project 1/Reconstruction-Wheat/processed_rgb_images'

# Instantiate dataset with your previously computed min/max
dataset = HSI_RGB_Dataset(new_hsi_save_dir, new_rgb_dir, hsi_min, hsi_max, rgb_min, rgb_max, target_size=(512, 512))
full_dataloader = DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

# Dataset lengths for split: 60% train, 20% val, 20% test
total_len = len(dataset)
train_len = int(0.6 * total_len)
val_len = int(0.2 * total_len)
test_len = total_len - train_len - val_len

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_len, val_len, test_len])

# DataLoaders with custom collate_fn
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

In [ ]:
rgb_tensor, hsi_tensor, _ = dataset[420]  # note variable names match returned tensors
# For RGB tensor
total_elements_rgb = rgb_tensor.numel()
num_nans_rgb = torch.isnan(rgb_tensor).sum().item()
num_non_nans_rgb = total_elements_rgb - num_nans_rgb

print(f"RGB Tensor - Total elements: {total_elements_rgb}")
print(f"RGB Tensor - NaNs: {num_nans_rgb}")
print(f"RGB Tensor - Non-NaNs: {num_non_nans_rgb}")

# For HSI tensor
total_elements_hsi = hsi_tensor.numel()
num_nans_hsi = torch.isnan(hsi_tensor).sum().item()
num_non_nans_hsi = total_elements_hsi - num_nans_hsi

print(f"HSI Tensor - Total elements: {total_elements_hsi}")
print(f"HSI Tensor - NaNs: {num_nans_hsi}")
print(f"HSI Tensor - Non-NaNs: {num_non_nans_hsi}")

In [ ]:
# Option 1: Using a single sample from your dataset
rgb_tensor, hsi_tensor,_ = dataset[10]  # Get the first sample
print("HSI min:", hsi_tensor.min().item())
print("HSI max:", hsi_tensor.max().item())

In [ ]:
print("RGB min:", rgb_tensor.min().item())
print("RGB max:", rgb_tensor.max().item())

In [ ]:
hsi_tensor.shape

In [ ]:
rgb_tensor.shape

In [14]:
# Get a sample from your dataset
rgb_tensor, hsi_tensor, _ = dataset[0]  # Or any valid index

# Convert tensors from (C, H, W) to (H, W, C) NumPy arrays
rgb = rgb_tensor.permute(1, 2, 0).numpy()
hsi = hsi_tensor.permute(1, 2, 0).numpy()

In [ ]:
from spectral import imshow

imshow(rgb)  # for RGB image
imshow(hsi)  # for HSI image

In [16]:

import torch
import torch.nn as nn

# Default convolutional layer
def default_conv(in_channels, out_channels, kernel_size, bias=True):
    return nn.Conv2d(
        in_channels, out_channels, kernel_size,
        padding=(kernel_size // 2), bias=bias
    )

# Basic Block for building the model
class BasicBlock(nn.Sequential):
    def __init__(
        self, conv, in_channels, out_channels, kernel_size, stride=1, bias=False,
        bn=True, act=nn.ReLU(True)):

        m = [conv(in_channels, out_channels, kernel_size, bias=bias)]
        if bn:
            m.append(nn.BatchNorm2d(out_channels))
        if act is not None:
            m.append(act)

        super(BasicBlock, self).__init__(*m)

# Residual Block used for residual learning
class ResBlock(nn.Module):
    def __init__(
        self, conv, n_feats, kernel_size,
        bias=True, bn=False, act=nn.ReLU(True), res_scale=1):

        super(ResBlock, self).__init__()
        m = []
        for i in range(2):
            m.append(conv(n_feats, n_feats, kernel_size, bias=bias))
            if bn:
                m.append(nn.BatchNorm2d(n_feats))
            if i == 0:
                m.append(act)

        self.body = nn.Sequential(*m)
        self.res_scale = res_scale

    def forward(self, x):
        res = self.body(x).mul(self.res_scale)
        res += x

        return res

# EDSR Model for hyperspectral image reconstruction
class EDSR(nn.Module):
    def __init__(self, conv=default_conv):
        super(EDSR, self).__init__()

        n_resblocks = 32  # Number of residual blocks
        n_feats = 64  # Number of feature channels
        kernel_size = 3  # Kernel size for convolution
        n_colors = 3  # Input RGB channels (fixed for RGB input)
        out_channels = 10  # Output channels set to 10 for hyperspectral reconstruction
        act = nn.ReLU(True)  # Activation function (ReLU)

        # Define head module (initial convolution)
        m_head = [conv(n_colors, n_feats, kernel_size)]

        # Define body module (stack of residual blocks)
        m_body = [
            ResBlock(
                conv, n_feats, kernel_size, act=act, res_scale=1
            ) for _ in range(n_resblocks)
        ]
        m_body.append(conv(n_feats, n_feats, kernel_size))  # Final convolution in the body

        # Define tail module (final convolution to output hyperspectral image)
        m_tail = [
            conv(n_feats, out_channels, kernel_size)  # Output 10 spectral bands (hyperspectral)
        ]

        # Combine all modules into the full network
        self.head = nn.Sequential(*m_head)
        self.body = nn.Sequential(*m_body)
        self.tail = nn.Sequential(*m_tail)

    def forward(self, x):
        x = self.head(x)  # Pass through head module

        res = self.body(x)  # Pass through body module
        res += x  # Add residual connection

        x = self.tail(res)  # Pass through tail module to get the final hyperspectral image

        return x



In [17]:
import torch.nn as nn
import torch.optim as optim

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        """
        Combined L1 and MSE loss.
        alpha: Weight for MSE loss. (1 - alpha) will be the weight for L1 loss.
               Example: alpha=0.5 means equal weighting for MSE and L1.
        """
        super(CombinedLoss, self).__init__()
        self.mse_loss = nn.MSELoss()
        self.l1_loss = nn.L1Loss()
        self.alpha = alpha  # Adjust alpha to balance both losses

    def forward(self, outputs, targets):
        loss = self.alpha * self.mse_loss(outputs, targets) + (1 - self.alpha) * self.l1_loss(outputs, targets)
        return loss


In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler

# Loss function: L1 Loss (Mean Absolute Error)
criterion = nn.L1Loss()

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate EDSR model (EDSR must be defined above in the same script/session)
model = EDSR().to(device)

# Optimizer and learning rate scheduler
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)


In [ ]:
model

In [20]:

import os
import torch
import torch.nn as nn
import torch.optim as optim

def train_hsi_reconstruction(model, train_loader, val_loader, criterion, optimizer, num_epochs=50):
    # Define and ensure save path
    save_path = "/mnt/gpfs2_4m/scratch/aaol232/Project 1/models/best_hsi_model.pth"
    save_dir = os.path.dirname(save_path)
    os.makedirs(save_dir, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    best_val_loss = float("inf")
    best_model_state = None
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for rgb_images, hsi_targets, _ in train_loader:
            rgb_images, hsi_targets = rgb_images.to(device), hsi_targets.to(device)

            optimizer.zero_grad()
            outputs = model(rgb_images)
            loss = criterion(outputs, hsi_targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for rgb_images, hsi_targets, _ in val_loader:
                rgb_images, hsi_targets = rgb_images.to(device), hsi_targets.to(device)
                outputs = model(rgb_images)
                loss = criterion(outputs, hsi_targets)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")

    # Save the best model after all epochs
    torch.save(model.state_dict(), save_path)
    print(f"✅ Best model saved to: {save_path} (Final Val Loss: {avg_val_loss:.4f})")

    return train_losses, val_losses




In [21]:
import torch
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


In [ ]:
# Train the model for 100 epochs
train_losses, val_losses = train_hsi_reconstruction(model, train_loader, val_loader, criterion, optimizer, num_epochs=150)

In [ ]:
plt.plot(train_losses)


In [ ]:
import matplotlib.pyplot as plt
import os

# Set font sizes
plt.rcParams.update({
    'font.size': 15,
    'axes.titlesize': 15,
    'axes.labelsize': 15,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 15
})

# Create the plot
plt.figure(figsize=(8, 5))
plt.plot(train_losses, color='blue')  # removed marker='o'
plt.xlabel("Epoch")
plt.ylabel("Training loss (MRAE)")
#plt.title("Training Loss Curve")

# Save the plot
save_path = "/mnt/gpfs2_4m/scratch/vvv/Project 1/train_loss_curve.jpg"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
plt.tight_layout()
plt.savefig(save_path, format='jpg', dpi=300)
plt.show()

print(f"✅ Training loss plot saved to: {save_path}")


In [ ]:
plt.plot(val_losses)

In [26]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu
from skimage.morphology import remove_small_objects

def extract_roi_mask(hsi_image, min_size=500):
    """
    Extracts the ROI (corn region) mask from a reconstructed HSI image using Otsu's thresholding.
    
    :param hsi_image: Reconstructed hyperspectral image (10, H, W).
    :param min_size: Minimum size of ROI to be considered valid.
    :return: Binary mask (H, W) where 1 indicates ROI and 0 is background.
    """
    # Step 1: Convert 10-band HSI to grayscale by averaging across bands
    grayscale_image = np.mean(hsi_image, axis=0)  # Shape: (H, W)
    
    # Step 2: Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(grayscale_image, (5, 5), 0)
    
    # Step 3: Compute Otsu's threshold
    thresh_value = threshold_otsu(blurred)
    binary_mask = (blurred > thresh_value).astype(np.uint8)  # Convert to binary (0,1)

    # Step 4: Remove small noise regions
    binary_mask = remove_small_objects(binary_mask.astype(bool), min_size=min_size).astype(np.uint8)

    return binary_mask

In [ ]:
import torch
import os
import numpy as np

# Instantiate EDSR model (same config as during training)
model = EDSR()

# Load trained weights
model.load_state_dict(torch.load("/mnt/gpfs2_4m/scratch/aaol232/Project 1/models/best_hsi_model.pth"))

# Set to eval mode and move to device
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def save_reconstructed_hsi_with_separate_roi(
    model, dataloader, 
    full_hsi_save_dir="/mnt/gpfs2_4m/scratch/VVVVV/Project 1/Reconstruction-Wheat/full_reconstructed_hsi",
    hsi_save_dir="/mnt/gpfs2_4m/scratch/VVVV/Project 1/Reconstruction-Wheat/bandsonly_reconstructed_hsi", 
    rgb_save_dir="/mnt/gpfs2_4m/scratch/VVVVV/Project 1/Reconstruction-Wheat/bandsonly_reconstructed_rgb"
):
    """
    Processes each sample from the dataloader and does the following:
      1. Saves the full reconstructed HSI image (as output by the model) to full_hsi_save_dir.
      2. Extracts an ROI mask from the reconstructed HSI and computes the mean reflectance 
         (10-band vector) for that ROI, saving it to hsi_save_dir.
      3. Extracts an ROI mask from the original RGB image (from the dataloader) and computes 
         its mean reflectance (3-band vector), saving it to rgb_save_dir.
    
    Assumes no data augmentation is being used (so each sample name is unique).
    
    :param model: Trained model for HSI reconstruction.
    :param dataloader: Dataloader yielding (rgb_images, _, sample_names).
    :param full_hsi_save_dir: Directory for full reconstructed HSI images.
    :param hsi_save_dir: Directory for ROI-based mean reflectance vectors for HSI.
    :param rgb_save_dir: Directory for ROI-based mean reflectance vectors for RGB.
    """
    os.makedirs(full_hsi_save_dir, exist_ok=True)
    os.makedirs(hsi_save_dir, exist_ok=True)
    os.makedirs(rgb_save_dir, exist_ok=True)
    
    model.eval()
    
    with torch.no_grad():
        for batch_idx, (rgb_images, _, sample_names) in enumerate(dataloader):
            rgb_images = rgb_images.to(device)  # (B, 3, H, W)
            reconstructed_hsi = model(rgb_images).cpu().numpy()  # (B, 10, H, W)
            rgb_images = rgb_images.cpu().numpy()  # (B, 3, H, W)
            
            for i, sample_name in enumerate(sample_names):
                full_hsi_path = os.path.join(full_hsi_save_dir, f"{sample_name}.npy")
                np.save(full_hsi_path, reconstructed_hsi[i])
                
                hsi_roi_mask = extract_roi_mask(reconstructed_hsi[i])
                if np.sum(hsi_roi_mask) == 0:
                    print(f"⚠️ Warning: No ROI pixels found in HSI for {sample_name}, skipping ROI processing for HSI.")
                    continue
                hsi_roi_pixels = reconstructed_hsi[i][:, hsi_roi_mask > 0]
                mean_hsi = np.mean(hsi_roi_pixels, axis=1)
                hsi_save_path = os.path.join(hsi_save_dir, f"{sample_name}.npy")
                np.save(hsi_save_path, mean_hsi)
                
                rgb_roi_mask = extract_roi_mask(rgb_images[i])
                if np.sum(rgb_roi_mask) == 0:
                    print(f"⚠️ Warning: No ROI pixels found in RGB for {sample_name}, skipping ROI processing for RGB.")
                    continue
                rgb_roi_pixels = rgb_images[i][:, rgb_roi_mask > 0]
                mean_rgb = np.mean(rgb_roi_pixels, axis=1)
                rgb_save_path = os.path.join(rgb_save_dir, f"{sample_name}.npy")
                np.save(rgb_save_path, mean_rgb)
    
    print(f"✅ Full HSI images saved in '{full_hsi_save_dir}', ROI-based HSI means in '{hsi_save_dir}', and RGB means in '{rgb_save_dir}'.")


# Call the function with the pretrained model and full dataloader
save_reconstructed_hsi_with_separate_roi(
    model,
    full_dataloader,
    full_hsi_save_dir="/mnt/gpfs2_4m/scratch/VVVV/Project 1/Reconstruction-Wheat/full_reconstructed_hsi",
    hsi_save_dir="/mnt/gpfs2_4m/scratch/VVVV/Project 1/Reconstruction-Wheat/bandsonly_reconstructed_hsi",
    rgb_save_dir="/mnt/gpfs2_4m/scratch/VVVVV/Project 1/Reconstruction-Wheat/bandsonly_reconstructed_rgb"
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def visualize_roi_mask(hsi_image):
    """
    Visualizes the grayscale image and the corresponding binary mask.
    
    :param hsi_image: Reconstructed hyperspectral image (10, H, W).
    """
    # Extract ROI mask
    mask = extract_roi_mask(hsi_image)

    # Convert HSI to grayscale for visualization
    grayscale_image = np.mean(hsi_image, axis=0) 

    # Plot the grayscale image and extracted mask
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(grayscale_image, cmap='gray')
    axes[0].set_title("Grayscale HSI Image")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title("Extracted ROI Mask")
    axes[1].axis("off")

    plt.show()

# Load a reconstructed HSI file with corrected absolute path
hsi_image = np.load("/mnt/gpfs2_4m/scratch/aaol232/Project 1/Reconstruction-Wheat/full_reconstructed_hsi/WF-CF-1.3-Rep8.npy")

# Visualize ROI mask extraction
visualize_roi_mask(hsi_image)


In [166]:
import matplotlib.pyplot as plt
import os

def visualize_predictions(model, dataloader, device='cuda'):
    """
    Fetches one batch from `dataloader`, runs the model to get predicted HSI,
    and plots the average spectrum (over H and W) across 10 wavelengths
    for a few samples.
    """
    model.eval()
    
    # Grab one batch of data
    rgb_images, hsi_targets, _ = next(iter(dataloader))
    rgb_images = rgb_images.to(device)
    hsi_targets = hsi_targets.to(device)
    
    with torch.no_grad():
        # Forward pass to get predictions
        predicted_hsi = model(rgb_images)  

    # 1) Average each image across height and width -> shape: (B, 10)
    predicted_hsi_mean = predicted_hsi.mean(dim=(2, 3))  # average over H, W
    print(predicted_hsi_mean)
    hsi_targets_mean = hsi_targets.mean(dim=(2, 3))      # average over H, W
    print(hsi_targets_mean)
    
    # 2) Bring tensors back to CPU numpy for plotting
    predicted_hsi_mean = predicted_hsi_mean.cpu().numpy()
    hsi_targets_mean   = hsi_targets_mean.cpu().numpy()
    
    # Plot actual vs predicted spectra for a few samples
    num_samples = min(4, len(hsi_targets_mean))
    # These are the 10 wavelengths you're using (nm)
    wavelengths = [
        419.3237, 432.3083, 433.608, 435.558, 450.5249,
        470.0921, 473.3583, 477.2796, 769.7403, 989.8269
    ]
    # Set font sizes globally
    plt.rcParams.update({
        'font.size': 18,
        'axes.titlesize': 18,
        'axes.labelsize': 18,
        'xtick.labelsize': 18,
        'ytick.labelsize': 18,
        'legend.fontsize': 18
    })
    
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 4))
    # If there's only 1 sample, `axes` isn't a list; make it one for consistency
    if num_samples == 1:
        axes = [axes]
    
    for i in range(num_samples):
        axes[i].plot(wavelengths, hsi_targets_mean[i], label="Actual", color="blue", marker="o")
        axes[i].plot(wavelengths, predicted_hsi_mean[i], label="Predicted", color="red", linestyle="dashed", marker="x")
        axes[i].set_title(f"Sample {i+1}")
        axes[i].set_xlabel("Wavelength (nm)")
        axes[i].set_ylabel("Reflectance")
        axes[i].legend()

    plt.tight_layout()

    save_path = "/mnt/gpfs2_4m/scratch/VVVV/Project 1/predicted_vs_actual_spectra.jpg"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, format='jpg', dpi=300)
    plt.show()  # ✅ Show the figure

    print(f"✅ Plot saved to {save_path}")




In [ ]:
visualize_predictions(model, val_loader)

In [169]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

def visualize_hsi_side_by_side(model, val_loader, max_batches=30):
    model.eval()
    batch_count = 0

    # Set global font size
    plt.rcParams.update({
        'font.size': 16,
        'axes.titlesize': 16,
        'axes.labelsize': 16,
        'xtick.labelsize': 16,
        'ytick.labelsize': 16,
        'legend.fontsize': 16
    })

    save_dir = "/mnt/gpfs2_4m/scratch/aaol232/Project 1"
    os.makedirs(save_dir, exist_ok=True)

    with torch.no_grad():
        for rgb_images, hsi_targets, _ in val_loader:
            rgb_images = rgb_images.to(next(model.parameters()).device)
            outputs = model(rgb_images).cpu().numpy()
            rgb_images = rgb_images.cpu().numpy()
            hsi_targets = hsi_targets.cpu().numpy()

            for i in range(len(rgb_images)):
                if batch_count >= max_batches:
                    return

                pred = np.mean(outputs[i], axis=0)
                gt = np.mean(hsi_targets[i], axis=0)

                fig, axs = plt.subplots(1, 2, figsize=(10, 4))
                axs[0].imshow(gt, cmap='viridis')
                axs[0].set_title("Ground Truth HSI")
                axs[1].imshow(pred, cmap='viridis')
                axs[1].set_title("Predicted HSI")
                plt.suptitle(f"Sample {batch_count+1}", fontsize=16)

                plt.tight_layout()

                # Save figure
                save_path = os.path.join(save_dir, f"sample_{batch_count+1}.jpg")
                plt.savefig(save_path, format='jpg', dpi=300)
                plt.show()

                batch_count += 1



In [170]:
#visualize_predictions(model, val_loader)

In [ ]:
visualize_hsi_side_by_side(model, val_loader)

In [ ]:
import torch
import torch.nn.functional as F

def compute_mrae(pred, target, eps=1e-8):
    """
    Mean Relative Absolute Error:
    MRAE = mean( |pred - target| / (target + eps) )
    """
    pred = pred.float()
    target = target.float()
    rel_error = torch.abs(pred - target) / (target + eps)
    mrae = torch.mean(rel_error)
    return mrae.item()

def compute_rmse(pred, target):
    """
    Root Mean Squared Error:
    RMSE = sqrt( mean( (pred - target)^2 ) )
    """
    pred = pred.float()
    target = target.float()
    mse = F.mse_loss(pred, target, reduction='mean')
    rmse = torch.sqrt(mse)
    return rmse.item()

def compute_psnr(pred, target, max_val=1.0):
    """
    Peak Signal-to-Noise Ratio (PSNR):
    PSNR = 20 * log10(max_val) - 10 * log10(MSE)
    Assumes pred and target are in [0, max_val].
    """
    pred = pred.float()
    target = target.float()
    mse = F.mse_loss(pred, target, reduction='mean')
    psnr = 20 * torch.log10(torch.tensor(max_val)) - 10 * torch.log10(mse + 1e-8)
    return psnr.item()

def evaluate(model, dataloader, device='cuda', max_val=1.0):
    """
    Evaluate the model on a given dataloader and return average MRAE, RMSE, and PSNR.
    """
    model.eval()
    total_mrae, total_rmse, total_psnr = 0.0, 0.0, 0.0
    count = 0

    with torch.no_grad():
        for rgb_input, hsi_target, _ in dataloader:
            rgb_input = rgb_input.to(device)
            hsi_target = hsi_target.to(device)

            pred_hsi = model(rgb_input)

            total_mrae += compute_mrae(pred_hsi, hsi_target)
            total_rmse += compute_rmse(pred_hsi, hsi_target)
            total_psnr += compute_psnr(pred_hsi, hsi_target, max_val=max_val)

            count += 1

    avg_mrae = total_mrae / count
    avg_rmse = total_rmse / count
    avg_psnr = total_psnr / count

    return avg_mrae, avg_rmse, avg_psnr

# -----------------------------
# Evaluate on test dataset
# -----------------------------
avg_mrae_test, avg_rmse_test, avg_psnr_test = evaluate(model, test_loader, max_val=1.0)

print(f"Test MRAE: {avg_mrae_test:.4f}")
print(f"Test RMSE: {avg_rmse_test:.4f}")
print(f"Test PSNR: {avg_psnr_test:.2f} dB")

In [ ]:
import matplotlib.pyplot as plt

def visualize_predictions(model, dataloader, device='cuda', num_samples=6):
    """
    Fetches one batch from `dataloader`, runs the model to get predicted HSI,
    and plots the average spectrum (over H and W) across 10 wavelengths
    for a few samples.
    """
    model.eval()
    
    # Grab one batch of data
    rgb_images, hsi_targets, _ = next(iter(dataloader))
    rgb_images = rgb_images.to(device)
    hsi_targets = hsi_targets.to(device)
    
    with torch.no_grad():
        # Forward pass to get predictions
        predicted_hsi = model(rgb_images)  

    # 1) Average each image across height and width -> shape: (B, 10)
    predicted_hsi_mean = predicted_hsi.mean(dim=(2, 3))  # average over H, W
    hsi_targets_mean = hsi_targets.mean(dim=(2, 3))      # average over H, W
    
    # 2) Bring tensors back to CPU numpy for plotting
    predicted_hsi_mean = predicted_hsi_mean.cpu().numpy()
    hsi_targets_mean   = hsi_targets_mean.cpu().numpy()
    
    # Plot actual vs predicted spectra for a few samples
    num_samples = min(num_samples, len(hsi_targets_mean))  # Allow flexible number of samples
    wavelengths = [
        487.7464, 661.2103, 671.2596, 865.9986, 903.1627,
        905.2323, 937.0293, 938.4144, 957.8312, 998.1930
    ]
    
    # Determine grid size for subplots (more samples)
    rows = (num_samples + 1) // 2  # Compute rows dynamically
    cols = 2 if num_samples > 1 else 1  # Set columns to 2 if more than 1 sample
    
    # Adjust subplot grid size
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
    
    # Flatten the axes array to handle indexing consistently
    if num_samples == 1:
        axes = [axes]  # Single plot case
    else:
        axes = axes.flatten()  # Flatten for consistency if multiple plots
    
    for i in range(num_samples):
        ax = axes[i]
        ax.plot(
            wavelengths, hsi_targets_mean[i],
            label="Actual", color="blue", marker="o"
        )
        ax.plot(
            wavelengths, predicted_hsi_mean[i],
            label="Predicted", color="red", linestyle="dashed", marker="x"
        )
        ax.set_title(f"Sample {i+1}")
        ax.set_xlabel("Wavelength (nm)")
        ax.set_ylabel("Reflectance")
        ax.legend()

    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()

# Example call with more samples (6 samples in this case)
visualize_predictions(model, val_loader, num_samples=6)





In [ ]:
import numpy as np
import os

# Define the path to your new file
file_path = "/mnt/gpfs2_4m/scratch/aaol232/Project 1/Reconstruction-Wheat/bandsonly_reconstructed_hsi/WF-CF-8.5-Rep4.npy"

# Check if the file exists at the given path
if os.path.exists(file_path):
    # Load the .npy file
    data = np.load(file_path)
    
    # Display the content or the shape of the loaded data
    print("Data loaded successfully!")
    print(f"Shape of the data: {data.shape}")  # Prints the shape of the data
    
    # If you want to see the content (for small arrays)
    print("Content of the data:")
    print(data)  # This will display the entire array; for large arrays, consider using slicing
    
else:
    print(f"File not found: {file_path}")


